# VZ MIGRATION — AUDIT #4: HTML SITEMAP COMPLETENESS & ALLOCATION


# README — Audit #4: HTML Sitemap Allocation & Reconciliation

**Project:** Varsity Zone — Scorpion to WordPress Migration  
**Audit Date:** 9 November 2026  
**Script:** `vz_staging_audit4_html_sitemap_allocation.py`  
**Output:** `vz-staging-audit4-html-sitemap-allocation-09-Nov-2026.xlsx`

---

## 1. Purpose

Audit #4 verifies that expected migrated URLs are included in the correct HTML sitemaps on the WordPress staging website.

The audit compares the expected staging URL inventory against URLs extracted from live HTML sitemaps, identifying missing pages, incorrect sitemap allocations, duplicates and unexpected entries.

This is a **sitemap completeness and allocation audit**, not a page availability or HTTP response code audit.

## 2. Websites

**Production (Scorpion):**  
https://www.varsityzone.com

**Staging (WordPress):**  
https://varsityzonedev.wpenginepowered.com

The script retrieves HTML sitemaps directly from the WordPress staging website.

No locally saved HTML sitemap files are used as a fallback.

## 3. Relationship to Previous Audits

| Audit | Purpose |
|---|---|
| Audit #1 | Verify migration completeness and staging HTTP response codes |
| Audit #2 | Identify unexpected staging pages |
| Audit #3 | Reconcile expected URLs against XML sitemaps |
| Audit #4 | Reconcile expected URLs against HTML sitemaps |

Audit #4 follows the same reconciliation principles as Audit #3, replacing XML sitemap extraction with HTML sitemap extraction.

## 4. Required Input Files

Place the following files in the same directory as the Python script.

### A. Audit #3 Excel workbook

`vz-staging-audit3-xml-sitemap-allocation-09-Nov-2026.xlsx`

Provides the expected URL baseline for reconciliation.

**Important:** Audit #4 can only identify missing URLs that are present in this baseline. The baseline should therefore contain the complete expected staging URL equivalents derived from the production inventory.

### B. Rendered locations HTML

`vz-locations-rendered-html.html`

Used to discover the local franchise location folders.

This is the previously saved rendered HTML of the WordPress staging `/locations` page.

It is used for location discovery only, not as a fallback for HTML sitemap retrieval.

## 5. Python Dependencies

Required libraries:

- requests — HTTP retrieval
- beautifulsoup4 — HTML parsing
- openpyxl — Excel input and output

Install in Jupyter:

```python
%pip install requests beautifulsoup4 openpyxl
```

The script does not require `artifact_tool`.

## 6. HTML Sitemap Discovery

The script identifies two types of HTML sitemaps.

**National HTML sitemap:**

`https://varsityzonedev.wpenginepowered.com/site-map`

**Local HTML sitemaps:**

`https://varsityzonedev.wpenginepowered.com/{location}/site-map`

For example:

`https://varsityzonedev.wpenginepowered.com/springdale-ar/site-map`

Local location folders are derived from the rendered locations HTML file.

Only URLs ending in the expected `/site-map` path are considered sitemap sources.

Ordinary pages such as `/about-us`, `/contact-us` and their local equivalents are not classified as sitemaps.

## 7. Live Retrieval and Error Handling

The script retrieves each HTML sitemap from the live WordPress staging website.

The retrieval configuration includes:

- Default two-second delay between requests
- 30-second request timeout
- Up to three attempts for transient failures
- Exponential backoff
- Handling of HTTP 429 and server errors
- Rejection of redirects to unrelated pages

If a sitemap cannot be retrieved successfully, it is classified as `RETRIEVAL_FAILED`.

URLs associated with an unavailable sitemap must not automatically be classified as missing.

This distinction prevents temporary network or server failures from producing misleading migration exceptions.

## 8. URL Normalization

Before reconciliation, URLs are normalized to ensure equivalent URLs can be compared consistently.

Normalization includes:

- Standardizing production and staging hostnames
- Removing trailing slashes where appropriate
- Removing URL fragments
- Handling relative and absolute links
- Standardizing URL paths

For example:

Production:

`https://www.varsityzone.com/springdale-ar/cooling/ac-repair/`

Expected staging equivalent:

`https://varsityzonedev.wpenginepowered.com/springdale-ar/cooling/ac-repair`

These represent the same logical page for migration reconciliation.

## 9. Reconciliation Logic

The script compares each expected staging URL with the URLs extracted from the live HTML sitemaps.

### Test A — Sitemap inclusion

Is the expected staging URL present in an HTML sitemap?

If not, flag it as missing, provided the relevant sitemap was retrieved successfully.

### Test B — Correct allocation

Is the URL included in the appropriate sitemap?

National URLs should appear in the national HTML sitemap.

Local URLs should appear in the corresponding franchise location HTML sitemap.

### Test C — Duplicate entries

Is the same URL listed more than once, either within one sitemap or across multiple sitemaps?

Duplicate occurrences should be identified for investigation.

### Test D — Unexpected entries

Does an HTML sitemap contain a URL that is absent from the expected migration inventory?

These entries should be identified separately for review.

### Test E — Sitemap retrieval

Was each expected national or local HTML sitemap successfully retrieved?

Failed retrievals must be reported independently of URL inclusion failures.

## 10. Example Missing URL

**Production URL:**

`https://www.varsityzone.com/springdale-ar/cooling/ac-repair/`

**Expected staging URL:**

`https://varsityzonedev.wpenginepowered.com/springdale-ar/cooling/ac-repair`

**Expected sitemap:**

`https://varsityzonedev.wpenginepowered.com/springdale-ar/site-map`

If the staging URL is absent from the successfully retrieved local HTML sitemap, the audit should flag it as:

`MISSING`

This remains a sitemap exception even if the staging page itself returns HTTP 200.

The page may have migrated successfully but still be missing from the HTML sitemap.

## 11. Excel Output

The script generates:

`vz-staging-audit4-html-sitemap-allocation-09-Nov-2026.xlsx`

The workbook contains six worksheets.

| Worksheet | Description |
|---|---|
| Summary | Overall audit results and exception totals |
| Exceptions | URLs and sitemaps requiring investigation |
| Full URL Audit | URL-by-URL reconciliation results |
| Sitemap Inventory | Discovered sitemap URLs and retrieval outcomes |
| Raw Entries | Extracted sitemap links and their source sitemaps |
| Location Folders | Discovered local franchise folder inventory |

The **Exceptions** worksheet is the primary developer-facing output.

## 12. Developer Interpretation

| Exception | Interpretation | Recommended action |
|---|---|---|
| MISSING | Expected URL absent from its HTML sitemap | Add or restore the URL in the correct sitemap |
| Wrong allocation | URL appears in an incorrect sitemap | Correct the sitemap assignment |
| Duplicate | URL appears more than once | Investigate and remove unnecessary duplicates |
| Unexpected | URL is not in the expected baseline | Determine whether the page is legitimate |
| RETRIEVAL_FAILED | Sitemap could not be retrieved | Investigate availability and rerun |

Exception labels should be interpreted alongside the actual status values and details recorded in the workbook.

## 13. Running the Audit

1. Ensure the script and required input files are in the same working directory.
2. Install the required Python dependencies.
3. Confirm that the WordPress staging website is accessible.
4. Execute the Python script.
5. Allow the script to retrieve the national and local HTML sitemaps.
6. Open the generated Excel workbook.
7. Review the Summary and Exceptions worksheets.
8. Send actionable exceptions to the development team.

The script should be rerun after sitemap corrections have been deployed.

## 14. Limitations

**Baseline dependency:** The audit cannot identify a missing production-equivalent URL if that URL was omitted from the expected URL baseline.

**Live retrieval dependency:** Network failures, access restrictions or server errors can prevent individual sitemaps from being audited.

**HTML structure dependency:** The HTML parser must identify the actual sitemap content rather than unrelated navigation, footer or header links.

**No page-status verification:** Inclusion in an HTML sitemap does not prove that the linked page returns HTTP 200. HTTP response verification is handled separately by Audit #1.

**No XML validation:** XML sitemap reconciliation is handled by Audit #3.

## 15. Completion Criteria

Audit #4 can be considered passed when:

- All expected national and local HTML sitemaps are successfully retrieved.
- Every expected migrated URL appears in its correct HTML sitemap.
- No unresolved missing URL exceptions remain.
- No unresolved incorrect allocation exceptions remain.
- Duplicate and unexpected URL entries have been reviewed and resolved or approved.
- The final Excel report is retained as migration QA evidence.

---

## 16. Audit Governance

**Audit owner:** Migration QA / SEO  
**Remediation owner:** WordPress development team  
**Source environment:** Live WordPress staging website  
**Authoritative URL baseline:** Expected production-to-staging URL inventory  
**Primary deliverable:** Excel reconciliation and exception report

Audit #4 forms part of the Varsity Zone November 2026 migration validation framework and complements the page migration, unexpected page discovery and XML sitemap audits.

**Final objective:** Confirm that the HTML sitemap architecture accurately represents the expected migrated website, with every national and local URL assigned to the correct sitemap.



In [34]:
%pip install requests beautifulsoup4 openpyxl pandas lxml


In [35]:
# Run in Jupyter with: %run vz_staging_audit4_html_sitemap_allocation.py

%run vz_staging_audit4_html_sitemap_allocation.py

Expected URLs: 1224 | Local sitemaps: 26 | Baseline: vz-staging-audit3-xml-sitemap-allocation-09-Nov-2026.xlsx / Full URL Audit / expected staging url
[1/27] NATIONAL: OK (51 links)
[2/27] akron-oh: OK (55 links)
[3/27] alpharetta-ga: OK (38 links)
[4/27] arvada-co: OK (54 links)
[5/27] benbrook-tx: OK (65 links)
[6/27] cape-coral-fl: OK (45 links)
[7/27] chamblee-ga: OK (57 links)
[8/27] charlotte-nc: OK (30 links)
[9/27] clearwater-fl: OK (46 links)
[10/27] elmendorf-tx: OK (21 links)
[11/27] fishers-in: OK (56 links)
[12/27] franklin-tn: OK (46 links)
[13/27] frisco-tx: OK (119 links)
[14/27] galleria-uptown-tx: OK (34 links)
[15/27] katy-tx: OK (31 links)
[16/27] loveland-co: OK (47 links)
[17/27] mckinney-tx: OK (54 links)
[18/27] mooresville-nc: OK (68 links)
[19/27] north-central-san-antonio-tx: OK (27 links)
[20/27] north-phoenix-az: OK (44 links)
[21/27] omaha-ne: OK (14 links)
[22/27] roanoke-tx: OK (94 links)
[23/27] scottsdale-az: OK (93 links)
[24/27] southwest-florida-fl: